# 0. Setup

In [1]:
import re
import os, sys
import time
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from typing import Optional, List, Dict, Any

from pathlib import Path 
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
from shapely import wkb

# Add project root to Python path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
  sys.path.insert(0, str(project_root))

# Import config and setup
from config import setup_notebook, get_path
setup_notebook()

✓ Project root: /workspace/project_paaral
✓ Working directory: /workspace/project_paaral
✓ Python path updated


{'project_root': PosixPath('/workspace/project_paaral'),
 'data': PosixPath('/workspace/project_paaral/data'),
 'modules': PosixPath('/workspace/project_paaral/modules'),
 'notebooks': PosixPath('/workspace/project_paaral/notebooks'),
 'output': PosixPath('/workspace/project_paaral/output'),
 'psgc_shapefiles': PosixPath('/workspace/project_paaral/data/philippines-psgc-shapefiles/dist')}

In [2]:
!ls '.'

1.0-graph-generation-to-deterministic.ipynb  cache
2.0-discrete-choice-modeling.ipynb	     config
BENEFICIARY_PIVOT_OPTIMIZATION.md	     data
CLAUDE.md				     modules
DATA_QUIRKS.md				     notebooks
NOTEBOOK_UPDATES_SUMMARY.md		     output
README.md				     references
SCIPY_OPTIMIZATION_SUMMARY.md		     requirements.txt
UNIFIED_GR7_FLOWS_IMPLEMENTATION.md	     results


# 1. Load data

In [3]:
DATA = 'data'

In [4]:
fname = 'esc_beneficiaries.parquet'
fpath = os.path.join(DATA, 'processed', fname)

df_benef = pd.read_parquet(fpath)
print(df_benef.shape)

(2681668, 13)


In [5]:
display(df_benef.head(3))

,deped_school_id,lrn,esc_school_id,school_name,grade,billing_statement_number,esc_subsidy_amount,sheet_name,school_year,grade_level,lrn_validated,lrn_school_id,region
0,475511,133526130177,1603520,"Adiong Memorial College Foundation, Inc.",9,ESC-227153,9000.0,BARMM,SY 2022-2023,Grade 9,133526130177,133526,BARMM
1,475511,133536150012,1603520,"Adiong Memorial College Foundation, Inc.",7,ESC-227153,9000.0,BARMM,SY 2022-2023,Grade 7,133536150012,133536,BARMM
2,475511,133536150011,1603520,"Adiong Memorial College Foundation, Inc.",7,ESC-227153,9000.0,BARMM,SY 2022-2023,Grade 7,133536150011,133536,BARMM


# 2. Process data

## 2.1. Aggregate

In [6]:
rel_cols = [
    'lrn_school_id',
    'deped_school_id',
    'lrn',
    'grade',
    'school_year',
    'esc_subsidy_amount'
]
df_b = df_benef[rel_cols]

In [7]:
df_b.head(3)

,lrn_school_id,deped_school_id,lrn,grade,school_year,esc_subsidy_amount
0,133526,475511,133526130177,9,SY 2022-2023,9000.0
1,133536,475511,133536150012,7,SY 2022-2023,9000.0
2,133536,475511,133536150011,7,SY 2022-2023,9000.0


In [8]:
df_b['school_year'].unique()

['SY 2022-2023', 'SY 2023-2024', 'SY 2024-2025']
Categories (3, object): ['SY 2022-2023', 'SY 2023-2024', 'SY 2024-2025']

In [15]:
mask = (
    (df_b['school_year'] == 'SY 2023-2024')
    & (df_b['grade'] == '7')
)
df_g7 = df_b.loc[mask]
print(df_g7.shape)

(226075, 6)


In [13]:
agg = (
    df_b.groupby(['deped_school_id','lrn_school_id']).agg(
        {
            'lrn':'nunique',
        }
    )
    .reset_index()
    .rename(
        columns={
            'deped_school_id':'school_id_esc',
            'lrn_school_id':'school_id_from_lrn',
        }
    )
)
print(agg.shape)

(337406, 3)


In [14]:
display(agg)

,school_id_esc,school_id_from_lrn,lrn
0,0,403031,1
1,0,403032,3
2,0,403042,1
3,0,406385,1
4,0,406386,26
...,...,...,...
337401,497503,402928,1
337402,497503,418001,1
337403,497503,418024,1
337404,497503,418040,2


In [15]:
agg['school_id_esc'].unique()

<StringArray>
[     '0', '302762', '400001', '400002', '400003', '400004', '400006',
 '400007', '400008', '400010',
 ...
 '495017', '495022', '495026', '495027', '495028', '495033', '495501',
 '496008', '496503', '497503']
Length: 3706, dtype: string

In [16]:
agg['school_id_from_lrn'].unique()

array(['403031', '403032', '403042', ..., '128747', '406608', '127976'],
      shape=(45896,), dtype=object)

## 2.2. Validate

**Student Flow Direction: ORIGIN → DESTINATION**
- **ORIGIN schools** (`school_id_from_lrn`): Where students came FROM (could be public or private)
- **DESTINATION schools** (`school_id_esc`): Where students GO TO (ESC recipient private schools)

**Validation Plan:**
1. Validate DESTINATION schools (school_id_esc) against private node table
2. Validate ORIGIN schools (school_id_from_lrn) against BOTH public and private node tables
3. Tag rows where both schools exist in our node tables

In [8]:
# Load valid node tables
public_nodes = gpd.read_file('output/public_nodes_valid.gpkg')
private_nodes = gpd.read_file('output/private_nodes_valid.gpkg')

print(f"Public nodes loaded: {len(public_nodes):,} schools")
print(f"Private nodes loaded: {len(private_nodes):,} schools")

Public nodes loaded: 44,899 schools
Private nodes loaded: 9,305 schools


In [9]:
# Extract valid school IDs as sets for fast lookup
valid_public_ids = set(public_nodes['school_id'].astype(str))
valid_private_ids = set(private_nodes['school_id'].astype(str))

print(f"Valid public school IDs: {len(valid_public_ids):,}")
print(f"Valid private school IDs: {len(valid_private_ids):,}")

Valid public school IDs: 44,899
Valid private school IDs: 9,275


In [10]:
# Ensure school IDs are strings for matching
agg['school_id_esc'] = agg['school_id_esc'].astype(str)
agg['school_id_from_lrn'] = agg['school_id_from_lrn'].astype(str)

# === VALIDATE DESTINATION SCHOOLS (where students GO TO) ===
# ESC recipient schools should be private schools
agg['esc_in_private_nodes'] = agg['school_id_esc'].isin(valid_private_ids)

# === VALIDATE ORIGIN SCHOOLS (where students CAME FROM) ===
# Origin schools could be either public or private
agg['origin_in_public_nodes'] = agg['school_id_from_lrn'].isin(valid_public_ids)
agg['origin_in_private_nodes'] = agg['school_id_from_lrn'].isin(valid_private_ids)

# Origin school is valid if found in EITHER public OR private nodes
agg['origin_school_valid'] = agg['origin_in_public_nodes'] | agg['origin_in_private_nodes']

# Both schools valid (complete edge: ORIGIN → DESTINATION)
agg['both_schools_valid'] = agg['esc_in_private_nodes'] & agg['origin_school_valid']

print("Validation complete!")
print(f"Total rows: {len(agg):,}")
print(f"\nDESTINATION school validation (ESC recipients):")
print(f"  In private nodes: {agg['esc_in_private_nodes'].sum():,} ({agg['esc_in_private_nodes'].sum()/len(agg)*100:.1f}%)")
print(f"\nORIGIN school validation (where students came from):")
print(f"  In public nodes: {agg['origin_in_public_nodes'].sum():,} ({agg['origin_in_public_nodes'].sum()/len(agg)*100:.1f}%)")
print(f"  In private nodes: {agg['origin_in_private_nodes'].sum():,} ({agg['origin_in_private_nodes'].sum()/len(agg)*100:.1f}%)")
print(f"  Valid (either): {agg['origin_school_valid'].sum():,} ({agg['origin_school_valid'].sum()/len(agg)*100:.1f}%)")
print(f"\nComplete edges (ORIGIN → DESTINATION both valid): {agg['both_schools_valid'].sum():,} ({agg['both_schools_valid'].sum()/len(agg)*100:.1f}%)")

NameError: name 'agg' is not defined

In [29]:
# Show breakdown of origin school types
print("Origin school type breakdown:")
print(f"  Public only: {(agg['origin_in_public_nodes'] & ~agg['origin_in_private_nodes']).sum():,}")
print(f"  Private only: {(~agg['origin_in_public_nodes'] & agg['origin_in_private_nodes']).sum():,}")
print(f"  Both (same ID in both datasets): {(agg['origin_in_public_nodes'] & agg['origin_in_private_nodes']).sum():,}")
print(f"  Neither: {(~agg['origin_school_valid']).sum():,}")

# Show some examples of valid complete edges
print("\n" + "="*60)
print("Sample of valid complete edges (both schools in node tables):")
print("="*60)
display(agg[agg['both_schools_valid']].head(10))

Origin school type breakdown:
  Public only: 177,181
  Private only: 92,625
  Both (same ID in both datasets): 0
  Neither: 67,600

Sample of valid complete edges (both schools in node tables):


,school_id_esc,school_id_from_lrn,lrn,esc_in_private_nodes,origin_in_public_nodes,origin_in_private_nodes,origin_school_valid,both_schools_valid
200,400006,100043,7,True,True,False,True,True
201,400006,100045,3,True,True,False,True,True
202,400006,100046,1,True,True,False,True,True
203,400006,100047,16,True,True,False,True,True
204,400006,100048,2,True,True,False,True,True
205,400006,100050,1,True,True,False,True,True
206,400006,100051,2,True,True,False,True,True
207,400006,100052,1,True,True,False,True,True
208,400006,100053,1,True,True,False,True,True
209,400006,100055,1,True,True,False,True,True


In [30]:
# Analyze unique schools
print("Unique school analysis:")
print(f"\nTotal unique ESC schools: {agg['school_id_esc'].nunique():,}")
print(f"  Matched in private nodes: {agg[agg['esc_in_private_nodes']]['school_id_esc'].nunique():,}")
print(f"  Not matched: {agg[~agg['esc_in_private_nodes']]['school_id_esc'].nunique():,}")

print(f"\nTotal unique origin schools: {agg['school_id_from_lrn'].nunique():,}")
print(f"  Matched in public nodes: {agg[agg['origin_in_public_nodes']]['school_id_from_lrn'].nunique():,}")
print(f"  Matched in private nodes: {agg[agg['origin_in_private_nodes']]['school_id_from_lrn'].nunique():,}")
print(f"  Matched in either: {agg[agg['origin_school_valid']]['school_id_from_lrn'].nunique():,}")
print(f"  Not matched: {agg[~agg['origin_school_valid']]['school_id_from_lrn'].nunique():,}")

# Create filtered dataset with only valid edges for graph building
agg_valid = agg[agg['both_schools_valid']].copy()
print(f"\n" + "="*60)
print(f"Filtered dataset ready for graph building:")
print(f"  Total valid edges: {len(agg_valid):,}")
print(f"  Unique origin schools: {agg_valid['school_id_from_lrn'].nunique():,}")
print(f"  Unique destination schools: {agg_valid['school_id_esc'].nunique():,}")
print("="*60)

Unique school analysis:

Total unique ESC schools: 3,706
  Matched in private nodes: 2,955
  Not matched: 751

Total unique origin schools: 45,896
  Matched in public nodes: 29,886
  Matched in private nodes: 7,071
  Matched in either: 36,957
  Not matched: 8,939

Filtered dataset ready for graph building:
  Total valid edges: 220,777
  Unique origin schools: 34,411
  Unique destination schools: 2,955


In [31]:
agg_valid

,school_id_esc,school_id_from_lrn,lrn,esc_in_private_nodes,origin_in_public_nodes,origin_in_private_nodes,origin_school_valid,both_schools_valid
200,400006,100043,7,True,True,False,True,True
201,400006,100045,3,True,True,False,True,True
202,400006,100046,1,True,True,False,True,True
203,400006,100047,16,True,True,False,True,True
204,400006,100048,2,True,True,False,True,True
...,...,...,...,...,...,...,...,...
337401,497503,402928,1,True,False,True,True,True
337402,497503,418001,1,True,False,True,True,True
337403,497503,418024,1,True,False,True,True,True
337404,497503,418040,2,True,False,True,True,True


# 3. Using BeneficiaryProcessor Module

Now let's use the newly created module to process the data more cleanly.

**Reminder - Student Flow Direction: ORIGIN → DESTINATION**
- `school_id_origin` (from `lrn_school_id`): Where students came FROM
- `school_id_destination` (from `deped_school_id`/`school_id_esc`): Where students GO TO (ESC recipients)

In [4]:
from modules.beneficiary_processor import BeneficiaryProcessor
import importlib
import modules.beneficiary_processor as bp

# Reload module
importlib.reload(bp)
from modules.beneficiary_processor import BeneficiaryProcessor

In [5]:
# Initialize processor
processor = BeneficiaryProcessor(
    public_nodes_path='output/public_nodes_valid.gpkg',
    private_nodes_path='output/private_nodes_valid.gpkg',
    verbose=True
)

# Run complete pipeline
valid_edges = processor.process(
    beneficiary_path=os.path.join(DATA, 'processed', 'esc_beneficiaries.parquet'),
    export_path='output/beneficiary_edges_valid.csv'
)

INFO:modules.beneficiary_processor:BeneficiaryProcessor initialized
INFO:modules.beneficiary_processor:============================================================
INFO:modules.beneficiary_processor:Starting beneficiary processing pipeline
INFO:modules.beneficiary_processor:============================================================
INFO:modules.beneficiary_processor:Loading node tables...
INFO:modules.beneficiary_processor:  Loaded 44,899 valid public schools
INFO:modules.beneficiary_processor:  Loaded 9,305 valid private schools
INFO:modules.beneficiary_processor:Loading beneficiary data from data/processed/esc_beneficiaries.parquet...
INFO:modules.beneficiary_processor:  Loading beneficiary data...
INFO:modules.beneficiary_processor:  Loaded 2,681,668 raw beneficiary records
INFO:modules.beneficiary_processor:  Aggregating beneficiary flows by origin-destination-year...
INFO:modules.beneficiary_processor:  Aggregated to 692,784 unique origin-destination-year triplets
INFO:modules.b

In [13]:
valid_edges.head()

school_year,school_id_destination,school_id_origin,beneficiaries_sy_2022_2023,beneficiaries_sy_2023_2024,beneficiaries_sy_2024_2025,total_beneficiaries,destination_in_private_nodes,origin_in_public_nodes,origin_in_private_nodes,origin_valid,both_schools_valid
202,400006,100043,4.0,7.0,5.0,16.0,True,True,False,True,True
203,400006,100045,2.0,3.0,2.0,7.0,True,True,False,True,True
204,400006,100046,1.0,1.0,NaN,2.0,True,True,False,True,True
205,400006,100047,10.0,9.0,12.0,31.0,True,True,False,True,True
206,400006,100048,1.0,NaN,1.0,2.0,True,True,False,True,True


In [11]:
# Get comprehensive summary
summary = processor.get_summary()

print("\n" + "="*60)
print("BENEFICIARY DATA SUMMARY")
print("Student Flow: ORIGIN → DESTINATION")
print("="*60)
print(f"\nTotal edges: {summary['total_edges']:,}")
print(f"Valid edges: {summary['valid_edges']:,} ({summary['validation_rate']:.1f}%)")

print(f"\nDESTINATION schools (ESC recipients - where students GO TO):")
print(f"  Matched in private nodes: {summary['destination_validation']['in_private_nodes']:,} ({summary['destination_validation']['percentage']:.1f}%)")

print(f"\nORIGIN schools (where students CAME FROM):")
print(f"  In public nodes: {summary['origin_validation']['in_public_nodes']:,}")
print(f"  In private nodes: {summary['origin_validation']['in_private_nodes']:,}")
print(f"  Valid (either): {summary['origin_validation']['valid_either']:,} ({summary['origin_validation']['percentage']:.1f}%)")

print(f"\nOrigin school types:")
print(f"  Public only: {summary['origin_type_breakdown']['public_only']:,}")
print(f"  Private only: {summary['origin_type_breakdown']['private_only']:,}")
print(f"  Both datasets: {summary['origin_type_breakdown']['both_datasets']:,}")
print(f"  Neither: {summary['origin_type_breakdown']['neither']:,}")

print(f"\nUnique schools:")
print(f"  Origins (total): {summary['unique_schools']['total_origins']:,}")
print(f"  Origins (matched): {summary['unique_schools']['origins_matched']:,}")
print(f"  Destinations (total): {summary['unique_schools']['total_destinations']:,}")
print(f"  Destinations (matched): {summary['unique_schools']['destinations_matched']:,}")

print(f"\nBeneficiary statistics:")
print(f"  Total beneficiaries: {summary['beneficiary_stats']['total_beneficiaries']:,}")
print(f"  Valid beneficiaries: {summary['beneficiary_stats']['valid_beneficiaries']:,}")
print(f"  Mean per edge: {summary['beneficiary_stats']['mean_per_edge']:.1f}")
print(f"  Median per edge: {summary['beneficiary_stats']['median_per_edge']:.1f}")
print("="*60)


BENEFICIARY DATA SUMMARY
Student Flow: ORIGIN → DESTINATION

Total edges: 337,992
Valid edges: 217,625 (64.4%)

DESTINATION schools (ESC recipients - where students GO TO):
  Matched in private nodes: 268,771 (79.5%)

ORIGIN schools (where students CAME FROM):
  In public nodes: 175,996
  In private nodes: 92,192
  Valid (either): 268,188 (79.3%)

Origin school types:
  Public only: 175,996
  Private only: 92,192
  Both datasets: 0
  Neither: 69,804

Unique schools:
  Origins (total): 45,897
  Origins (matched): 36,628
  Destinations (total): 3,706
  Destinations (matched): 2,928

Beneficiary statistics:
  Total beneficiaries: 2,681,651.0
  Valid beneficiaries: 1,840,387.0
  Mean per edge: 7.9
  Median per edge: 2.0


In [12]:
# Display sample of valid edges
print("Sample of valid edges:")
display(valid_edges.head(10))

Sample of valid edges:


school_year,school_id_destination,school_id_origin,beneficiaries_sy_2022_2023,beneficiaries_sy_2023_2024,beneficiaries_sy_2024_2025,total_beneficiaries,destination_in_private_nodes,origin_in_public_nodes,origin_in_private_nodes,origin_valid,both_schools_valid
202,400006,100043,4.0,7.0,5.0,16.0,True,True,False,True,True
203,400006,100045,2.0,3.0,2.0,7.0,True,True,False,True,True
204,400006,100046,1.0,1.0,NaN,2.0,True,True,False,True,True
205,400006,100047,10.0,9.0,12.0,31.0,True,True,False,True,True
206,400006,100048,1.0,NaN,1.0,2.0,True,True,False,True,True
207,400006,100050,NaN,1.0,NaN,1.0,True,True,False,True,True
208,400006,100051,2.0,2.0,1.0,5.0,True,True,False,True,True
209,400006,100052,NaN,NaN,1.0,1.0,True,True,False,True,True
210,400006,100053,NaN,NaN,1.0,1.0,True,True,False,True,True
211,400006,100055,1.0,1.0,1.0,3.0,True,True,False,True,True


## 3.1. Provincial Filtering Example

Test filtering to a specific province (Bulacan - PH03014)

In [7]:
# # Filter to Bulacan province
# bulacan_edges = processor.filter_by_province('PH03014', valid_only=True)

# print(f"\nBulacan beneficiary edges: {len(bulacan_edges):,}")
# print(f"Unique origins: {bulacan_edges['school_id_origin'].nunique():,}")
# print(f"Unique destinations: {bulacan_edges['school_id_destination'].nunique():,}")
# print(f"Total beneficiaries: {bulacan_edges['beneficiary_count'].sum():,}")

# print("\nSample Bulacan edges:")
# display(bulacan_edges.head())